# HW6. Замеры инференса Qwen3-8B

На GPU-сервере выполните `bash setup_env.sh`, выберите ядро **Python (hw6_safin)** и запустите **Run All**. Ноутбук поднимет vLLM, выполнит три этапа и сохранит результаты в отдельную папку `results/`.

In [3]:
import os

from hw6_experiments import Experiment
from hw6_results import write_results

GPU_DEVICE = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
SPEC_RPMS = [60, 120, 240, 480, 960, 1920]
MAX_CAPACITY_RPM = 3840

experiment = Experiment(GPU_DEVICE, SPEC_RPMS, MAX_CAPACITY_RPM)

NVIDIA A100-SXM4-80GB (81920 MiB). Результаты: /home/edu/amsafin/HSE-Avito/LLM/hw6/results/20261119-225250-981491


## 1. Подбор конфигурации при 120 RPM

Сравниваем дефолтный vLLM, настройки шедулера и n-gram. Победителя выбираем по минимальной средней E2E latency, при равенстве по P90.

In [ ]:
try:
    winner = experiment.run_fixed()
    print(f"{winner['config']}: {winner['speedup_vs_baseline']:.3f}× к бейзлайну")
finally:
    results_notebook = write_results(experiment.directory)
    print(results_notebook)

fixed / baseline: запуск vLLM


## 2. Влияние нагрузки на n-gram

Сравниваем дефолтный vLLM и его вариант с лучшим n-gram при 120 RPM. Остальные параметры движка совпадают. Для каждой нагрузки измеряем P90 на всех исходных запросах.

In [ ]:
try:
    experiment.run_spec()
finally:
    results_notebook = write_results(experiment.directory)
    print(results_notebook)

## 3. Поиск границы SLA

На победителе проверяем нагрузки длительностью 300 секунд. Поиск уточняет границу с шагом 15 RPM. Для SLA требуются P90 E2E строго меньше 15 секунд и ответы на все поданные запросы.

In [ ]:
try:
    experiment.run_capacity()
finally:
    results_notebook = write_results(experiment.directory)
    print(results_notebook)